# RAL-CLIP from proposal

Prototype notebook cho **Real-Anchored Local CLIP Adaptation (RAL-CLIP)** trong `proposal/Training_free_TTA_for_image_deepfake_detection (1).pdf`.

Khác notebook baseline trước đó, notebook này chạy từ **image paths** để lấy CLIP global embeddings và **local patch tokens**. Pipeline: source real-only memory -> semantic retrieval -> local real-deviation -> top-K pooling -> optional safe target real expansion -> unsupervised GMM threshold calibration.

In [ ]:
from pathlib import Path
import math
import sys
import warnings

import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from PIL import Image
from sklearn.metrics import accuracy_score, average_precision_score, confusion_matrix, f1_score, roc_auc_score, roc_curve
from sklearn.mixture import GaussianMixture
from torch.utils.data import DataLoader, Dataset
from tqdm.auto import tqdm

def find_repo_root():
    for root in [Path.cwd(), *Path.cwd().parents, Path('/kaggle/working/training-free-tta-for-deepfake-detection')]:
        if (root / 'code' / 'deepfake_tta').exists():
            return root.resolve()
    raise FileNotFoundError('Cannot find repo root containing code/deepfake_tta')

REPO_ROOT = find_repo_root()
CODE_ROOT = REPO_ROOT / 'code'
sys.path.insert(0, str(CODE_ROOT))

from training.ffpp_split_utils import prepare_ffpp_split_dataframe

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('repo:', REPO_ROOT)
print('device:', DEVICE)

## Config

Sửa các path bên dưới theo Kaggle/local. `SOURCE_DF` lấy FF++ train rồi chỉ giữ label REAL để build memory. `TARGET_SPECS` có thể là clean dataset hoặc corrupted dataset đã remap root.

In [ ]:
ON_KAGGLE = Path('/kaggle/input').exists()

if ON_KAGGLE:
    CSV_PATH = Path('/kaggle/input/datasets/jamestashvik/deepfakebench/deepfakebench_dataset.csv')
    DEEPFAKEBENCH_ROOT = Path('/kaggle/input/datasets/jamestashvik/deepfakebench/DeepFakeBench')
    SPLIT_ROOT = None
    OUTPUT_DIR = Path('/kaggle/working/ral_clip')
else:
    CSV_PATH = REPO_ROOT / 'deepfakebench_dataset.csv'
    DEEPFAKEBENCH_ROOT = REPO_ROOT / 'DeepFakeBench'
    SPLIT_ROOT = None
    OUTPUT_DIR = REPO_ROOT / 'eda' / 'ral_clip'

CLIP_MODEL = 'ViT-L-14'
PRETRAINED = 'openai'
LAYERS = [-6]          # intermediate CLIP layers. Try [-8, -6, -4] for ablation.
LAYER_WEIGHTS = None   # None = equal weights.

MAX_SOURCE_REAL = 2048 # memory size. Increase on GPU if VRAM allows.
MAX_TARGET = 2000      # set None for full target set.
BATCH_SIZE = 16
NUM_WORKERS = 2
USE_AMP = True
SEED = 42

RETRIEVE_M = 16
TOP_K_PATCHES = 16
NEIGHBOR_RADIUS = 1
RUN_TARGET_EXPANSION = True
EXPAND_SCORE_QUANTILE = 0.20
EXPAND_MAX_AUG_VAR = 1e-4
MAX_TARGET_REAL_ADD = 512

TARGET_SPECS = [
    {'name': 'celebdfv1-clean', 'dataset_name': 'Celeb-DF-v1', 'root': DEEPFAKEBENCH_ROOT},
    # Example corrupted target. Point root to folder containing Celeb-real/YouTube-real/Celeb-synthesis.
    # {'name': 'celebdfv1-gaussian_blur-l2', 'dataset_name': 'Celeb-DF-v1', 'root': Path('/kaggle/input/celebdfv1-corruption-level-2/gaussian_blur/level_2')},
    # {'name': 'ffpp-test-clean', 'dataset_name': 'FaceForensics++', 'root': DEEPFAKEBENCH_ROOT, 'ffpp_split': 'test'},
]

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
torch.manual_seed(SEED)
np.random.seed(SEED)

In [ ]:
def prepare_deepfakebench_dataframe(csv_path, dataset_name, deepfakebench_root, *, replacement_root=None):
    df = pd.read_csv(csv_path)
    df = df[df['datasetname'].eq(dataset_name)].copy()
    if df.empty:
        raise ValueError(f'No rows for datasetname={dataset_name!r}')
    df['label_num'] = df['label'].map({'REAL': 0, 'FAKE': 1}).astype(int)
    df['imagepath_fixed'] = df['imagepath'].astype(str).str.replace('../input/deepfakebench', str(deepfakebench_root), regex=False)
    if replacement_root is not None:
        source_root = Path(deepfakebench_root) / dataset_name
        df['imagepath_fixed'] = df['imagepath_fixed'].astype(str).str.replace(str(source_root), str(replacement_root), regex=False)
    exists = df['imagepath_fixed'].map(lambda p: Path(p).exists())
    missing = int((~exists).sum())
    if missing:
        print(f'dropping {missing}/{len(df)} missing files for {dataset_name}')
        print(df.loc[~exists, ['imagepath', 'imagepath_fixed']].head(5).to_string(index=False))
        df = df[exists].copy()
    print(dataset_name, df.shape, df['label_num'].value_counts().sort_index().to_dict())
    return df.reset_index(drop=True)

def maybe_sample_df(df, max_rows, seed, *, balanced=False):
    if max_rows is None or len(df) <= max_rows:
        return df.reset_index(drop=True)
    if balanced:
        per_class = max_rows // 2
        parts = []
        for label in [0, 1]:
            sub = df[df['label_num'].eq(label)]
            parts.append(sub.sample(n=min(per_class, len(sub)), random_state=seed))
        return pd.concat(parts).sample(frac=1, random_state=seed).reset_index(drop=True)
    return df.sample(n=max_rows, random_state=seed).reset_index(drop=True)

class PathImageDataset(Dataset):
    def __init__(self, dataframe, preprocess, augment=None):
        self.df = dataframe.reset_index(drop=True)
        self.preprocess = preprocess
        self.augment = augment
    def __len__(self):
        return len(self.df)
    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        image = Image.open(row['imagepath_fixed']).convert('RGB')
        if self.augment == 'hflip':
            image = image.transpose(Image.Transpose.FLIP_LEFT_RIGHT)
        return self.preprocess(image), int(row['label_num']), row['imagepath_fixed']

def build_source_real_df():
    source = prepare_ffpp_split_dataframe(CSV_PATH, DEEPFAKEBENCH_ROOT, split_name='train', split_root=SPLIT_ROOT)
    source = source[source['label_num'].eq(0)].reset_index(drop=True)
    source = maybe_sample_df(source, MAX_SOURCE_REAL, SEED, balanced=False)
    print('source real memory rows:', len(source))
    return source

def build_target_df(spec):
    if spec.get('dataset_name') == 'FaceForensics++' and spec.get('ffpp_split'):
        df = prepare_ffpp_split_dataframe(CSV_PATH, DEEPFAKEBENCH_ROOT, split_name=spec['ffpp_split'], split_root=SPLIT_ROOT)
        if Path(spec['root']) != Path(DEEPFAKEBENCH_ROOT):
            source_root = Path(DEEPFAKEBENCH_ROOT) / 'FaceForensics++'
            df['imagepath_fixed'] = df['imagepath_fixed'].astype(str).str.replace(str(source_root), str(spec['root']), regex=False)
    else:
        root = spec.get('root', DEEPFAKEBENCH_ROOT)
        replacement = None if Path(root) == Path(DEEPFAKEBENCH_ROOT) else root
        df = prepare_deepfakebench_dataframe(CSV_PATH, spec['dataset_name'], DEEPFAKEBENCH_ROOT, replacement_root=replacement)
    return maybe_sample_df(df, MAX_TARGET, SEED + 17, balanced=True)

In [ ]:
class OpenClipLocalExtractor:
    def __init__(self, model, layers):
        self.model = model.eval()
        self.layers = list(layers)
        self._captures = {}
        self._handles = []
        blocks = self._resblocks()
        n = len(blocks)
        self.layer_indices = [layer if layer >= 0 else n + layer for layer in self.layers]
        for idx in self.layer_indices:
            if idx < 0 or idx >= n:
                raise IndexError(f'Layer index {idx} out of range for {n} visual transformer blocks')
            self._handles.append(blocks[idx].register_forward_hook(self._make_hook(idx)))
        print('hooked visual layers:', self.layer_indices)
    def _resblocks(self):
        visual = self.model.visual
        if hasattr(visual, 'transformer') and hasattr(visual.transformer, 'resblocks'):
            return visual.transformer.resblocks
        if hasattr(visual, 'trunk') and hasattr(visual.trunk, 'blocks'):
            return visual.trunk.blocks
        raise TypeError('Cannot find OpenCLIP visual transformer blocks. Use a ViT OpenCLIP model.')
    def _make_hook(self, idx):
        def hook(module, inputs, output):
            x = output[0] if isinstance(output, tuple) else output
            self._captures[idx] = x.detach()
        return hook
    def close(self):
        for h in self._handles:
            h.remove()
        self._handles = []
    @torch.inference_mode()
    def __call__(self, images, use_amp=True):
        self._captures = {}
        with torch.autocast(device_type='cuda', dtype=torch.float16, enabled=(use_amp and images.device.type == 'cuda')):
            global_feats = self.model.encode_image(images)
        global_feats = F.normalize(global_feats.float(), dim=-1)
        locals_by_layer = []
        batch = images.shape[0]
        for idx in self.layer_indices:
            x = self._captures[idx].float()
            if x.ndim != 3:
                raise ValueError(f'Expected 3D token tensor, got {tuple(x.shape)}')
            if x.shape[0] == batch:
                tokens = x
            elif x.shape[1] == batch:
                tokens = x.permute(1, 0, 2).contiguous()
            else:
                raise ValueError(f'Cannot infer batch dimension from token shape {tuple(x.shape)}')
            # Drop CLS/prefix token. Patch tokens remain [B, N, D].
            patch_tokens = F.normalize(tokens[:, 1:, :], dim=-1)
            locals_by_layer.append(patch_tokens.cpu())
        return global_feats.cpu(), locals_by_layer

def create_clip_and_extractor():
    import open_clip
    model, _, preprocess = open_clip.create_model_and_transforms(CLIP_MODEL, pretrained=PRETRAINED, device=DEVICE)
    model.eval()
    for p in model.parameters():
        p.requires_grad_(False)
    return model, preprocess, OpenClipLocalExtractor(model, LAYERS)

In [ ]:
def extract_memory(dataframe, preprocess, extractor, *, augment=None, desc='extract'):
    loader = DataLoader(PathImageDataset(dataframe, preprocess, augment=augment), batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=torch.cuda.is_available())
    globals_out, labels_out, paths_out = [], [], []
    locals_out = None
    for images, labels, paths in tqdm(loader, desc=desc):
        images = images.to(DEVICE, non_blocking=True)
        g, locals_by_layer = extractor(images, use_amp=USE_AMP)
        globals_out.append(g)
        labels_out.append(labels.cpu())
        paths_out.extend(paths)
        if locals_out is None:
            locals_out = [[] for _ in locals_by_layer]
        for i, z in enumerate(locals_by_layer):
            locals_out[i].append(z)
    return {
        'global': torch.cat(globals_out).contiguous(),
        'local': [torch.cat(parts).contiguous() for parts in locals_out],
        'labels': torch.cat(labels_out).long(),
        'paths': paths_out,
    }

def local_deviation_one(target_z, real_z, neighbor_radius=1):
    # target_z: [N, D], real_z: [M, N, D]
    n = target_z.shape[0]
    side = int(round(math.sqrt(n)))
    if side * side != n:
        neighbor_radius = 0
    dists = []
    for i in range(n):
        if neighbor_radius > 0:
            r, c = divmod(i, side)
            js = []
            for rr in range(max(0, r - neighbor_radius), min(side, r + neighbor_radius + 1)):
                for cc in range(max(0, c - neighbor_radius), min(side, c + neighbor_radius + 1)):
                    js.append(rr * side + cc)
            candidates = real_z[:, js, :].reshape(-1, real_z.shape[-1])
        else:
            candidates = real_z[:, i, :]
        max_sim = torch.matmul(candidates, target_z[i]).max()
        dists.append(1.0 - max_sim)
    return torch.stack(dists)

def score_bank(target_bank, memory_bank, *, retrieve_m=16, top_k=16, neighbor_radius=1, layer_weights=None, desc='score'):
    mem_g = memory_bank['global']
    tgt_g = target_bank['global']
    sims = torch.matmul(tgt_g, mem_g.T)
    retrieve_m = min(retrieve_m, mem_g.shape[0])
    top_indices = sims.topk(k=retrieve_m, dim=1).indices
    if layer_weights is None:
        layer_weights = [1.0 / len(target_bank['local'])] * len(target_bank['local'])
    scores = []
    for row in tqdm(range(tgt_g.shape[0]), desc=desc):
        idx = top_indices[row]
        layer_scores = []
        for weight, target_local, mem_local in zip(layer_weights, target_bank['local'], memory_bank['local']):
            patch_d = local_deviation_one(target_local[row], mem_local[idx], neighbor_radius=neighbor_radius)
            k = min(top_k, patch_d.numel())
            layer_scores.append(float(weight) * float(patch_d.topk(k).values.mean()))
        scores.append(sum(layer_scores))
    return np.asarray(scores, dtype=np.float32)

def append_real_memory(memory_bank, target_bank, indices):
    if len(indices) == 0:
        return memory_bank
    idx = torch.as_tensor(indices, dtype=torch.long)
    memory_bank['global'] = torch.cat([memory_bank['global'], target_bank['global'][idx]], dim=0).contiguous()
    memory_bank['local'] = [torch.cat([m, t[idx]], dim=0).contiguous() for m, t in zip(memory_bank['local'], target_bank['local'])]
    memory_bank['labels'] = torch.cat([memory_bank['labels'], torch.zeros(len(idx), dtype=torch.long)], dim=0)
    memory_bank['paths'] = memory_bank['paths'] + [target_bank['paths'][int(i)] for i in idx]
    return memory_bank

In [ ]:
def gmm_threshold(scores):
    scores = np.asarray(scores, dtype=np.float64)
    if len(np.unique(scores)) < 3:
        return float(np.median(scores)), None
    gmm = GaussianMixture(n_components=2, random_state=SEED)
    gmm.fit(scores.reshape(-1, 1))
    means = gmm.means_.ravel()
    order = np.argsort(means)
    lo, hi = means[order[0]], means[order[1]]
    grid = np.linspace(scores.min(), scores.max(), 4096)
    logprob = gmm._estimate_weighted_log_prob(grid.reshape(-1, 1))
    diff = logprob[:, order[0]] - logprob[:, order[1]]
    between = (grid >= lo) & (grid <= hi)
    if between.any():
        idxs = np.where(between)[0]
        threshold = grid[idxs[np.argmin(np.abs(diff[idxs]))]]
    else:
        threshold = (lo + hi) / 2
    return float(threshold), gmm

def eer(y_true, y_score):
    fpr, tpr, thresholds = roc_curve(y_true, y_score)
    fnr = 1 - tpr
    i = np.nanargmin(np.abs(fnr - fpr))
    return float((fpr[i] + fnr[i]) / 2), float(thresholds[i])

def evaluate_scores(labels, scores, threshold, name):
    labels = np.asarray(labels).astype(int)
    preds = (np.asarray(scores) > threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(labels, preds, labels=[0, 1]).ravel()
    out = {
        'dataset': name,
        'method': 'ral_clip',
        'threshold': float(threshold),
        'acc': accuracy_score(labels, preds),
        'f1': f1_score(labels, preds, average='macro', zero_division=0),
        'auc': roc_auc_score(labels, scores) if len(np.unique(labels)) == 2 else np.nan,
        'ap': average_precision_score(labels, scores) if len(np.unique(labels)) == 2 else np.nan,
        'eer': eer(labels, scores)[0] if len(np.unique(labels)) == 2 else np.nan,
        'tn': int(tn), 'fp': int(fp), 'fn': int(fn), 'tp': int(tp),
    }
    return out

def save_score_table(name, target_bank, scores, threshold, metrics):
    table = pd.DataFrame({
        'dataset': name,
        'path': target_bank['paths'],
        'label': target_bank['labels'].numpy(),
        'score': scores,
        'pred': (scores > threshold).astype(int),
    })
    path = OUTPUT_DIR / f'{name}_ral_clip_scores.csv'
    table.to_csv(path, index=False)
    print('saved scores:', path)
    print(metrics)
    return path

In [ ]:
# Main run
model, preprocess, extractor = create_clip_and_extractor()
source_real_df = build_source_real_df()
source_memory = extract_memory(source_real_df, preprocess, extractor, desc='source real memory')

all_metrics = []
for spec in TARGET_SPECS:
    name = spec['name']
    print('\n===', name, '===')
    target_df = build_target_df(spec)
    target_bank = extract_memory(target_df, preprocess, extractor, desc=f'{name} target')
    memory = {k: (v.clone() if torch.is_tensor(v) else [x.clone() for x in v] if isinstance(v, list) and v and torch.is_tensor(v[0]) else list(v)) for k, v in source_memory.items()}
    scores = score_bank(target_bank, memory, retrieve_m=RETRIEVE_M, top_k=TOP_K_PATCHES, neighbor_radius=NEIGHBOR_RADIUS, layer_weights=LAYER_WEIGHTS, desc=f'{name} RAL score pass1')

    added = 0
    if RUN_TARGET_EXPANSION and len(target_df) > 0:
        hflip_bank = extract_memory(target_df, preprocess, extractor, augment='hflip', desc=f'{name} hflip for safe expansion')
        hflip_scores = score_bank(hflip_bank, memory, retrieve_m=RETRIEVE_M, top_k=TOP_K_PATCHES, neighbor_radius=NEIGHBOR_RADIUS, layer_weights=LAYER_WEIGHTS, desc=f'{name} hflip score')
        mean_scores = (scores + hflip_scores) / 2.0
        var_scores = np.var(np.stack([scores, hflip_scores], axis=1), axis=1)
        safe_tau = float(np.quantile(mean_scores, EXPAND_SCORE_QUANTILE))
        safe = np.where((mean_scores < safe_tau) & (var_scores < EXPAND_MAX_AUG_VAR))[0]
        safe = safe[:MAX_TARGET_REAL_ADD]
        memory = append_real_memory(memory, target_bank, safe)
        added = len(safe)
        print('safe target real added:', added, '| tau_real:', safe_tau)
        scores = score_bank(target_bank, memory, retrieve_m=RETRIEVE_M, top_k=TOP_K_PATCHES, neighbor_radius=NEIGHBOR_RADIUS, layer_weights=LAYER_WEIGHTS, desc=f'{name} RAL score pass2')

    threshold, _ = gmm_threshold(scores)
    metrics = evaluate_scores(target_bank['labels'].numpy(), scores, threshold, name)
    metrics.update({'target_samples': len(target_bank['labels']), 'source_real_memory': len(source_memory['labels']), 'target_real_added': added, 'retrieve_m': RETRIEVE_M, 'top_k_patches': TOP_K_PATCHES, 'neighbor_radius': NEIGHBOR_RADIUS, 'layers': str(LAYERS)})
    save_score_table(name, target_bank, scores, threshold, metrics)
    all_metrics.append(metrics)

summary = pd.DataFrame(all_metrics)
summary_path = OUTPUT_DIR / 'ral_clip_summary.csv'
summary.to_csv(summary_path, index=False)
display(summary)
print('saved summary:', summary_path)